# Análisis de video y estado del tráfico

Usá esta notebook cuando quieras aplicar un bundle de VAAET a un video y obtener estados de tráfico por minuto. El video siempre se anota; PostgreSQL y la revisión humana son opciones independientes y seguras por defecto.

| Necesitás | La notebook entrega | Requisito mínimo |
|---|---|---|
| Un MP4 + bundle válido de cuatro archivos | Video anotado, telemetría y estados | Dos minutos completos para el primer estado |
| PostgreSQL, sólo si decidís persistir | Features y predicciones idempotentes | Perfil `inference` |
| Revisión humana, sólo si la activás | Validaciones y paquete HITL | `VAAET_REVIEWER_ID` |

**Inicio rápido recomendado:** usá la configuración predeterminada. Carga un bundle piloto, no escribe en PostgreSQL y no abre revisión humana.

> Autorizar un bundle permite probarlo; **no lo promociona** ni cambia su manifiesto. Accident nunca es una salida automática.

## 1. Configuración central

Editá solamente la siguiente celda. Las demás consumen esas opciones sin redefinirlas. Los valores predeterminados ejecutan un piloto offline y conservador.

In [ ]:
# Workflow configuration — edit only this cell
ALLOW_PILOT_BUNDLE = True
ALLOW_EXPERIMENTAL_BUNDLE = False
PERSIST_TO_DATABASE = False
ENABLE_HUMAN_REVIEW = False
REVIEW_MODE = "priority"  # priority or all
DOWNLOAD_ANNOTATED_VIDEO = True
SHOW_DASHBOARD = True
HUD_DEBUG = False

if REVIEW_MODE not in {"priority", "all"}:
    raise ValueError("REVIEW_MODE debe ser 'priority' o 'all'.")

_allowed_bundles = ['production']
if ALLOW_PILOT_BUNDLE:
    _allowed_bundles.append('pilot')
if ALLOW_EXPERIMENTAL_BUNDLE:
    _allowed_bundles.append('candidate offline')
_review_destination = (
    'PostgreSQL + paquete HITL'
    if ENABLE_HUMAN_REVIEW and PERSIST_TO_DATABASE
    else 'paquete HITL portable'
    if ENABLE_HUMAN_REVIEW
    else 'sin revisión'
)
print("✅ Configuración validada")
print("🧭 Flujo seleccionado")
print(f"   Bundles permitidos: {', '.join(_allowed_bundles)}")
print(f"   PostgreSQL: {'activado' if PERSIST_TO_DATABASE else 'desactivado'}")
print(f"   Revisión humana: {REVIEW_MODE if ENABLE_HUMAN_REVIEW else 'desactivada'}")
print(f"   Destino HITL: {_review_destination}")
print(f"   Video: {'descarga automática' if DOWNLOAD_ANNOTATED_VIDEO else 'queda en /content'}")
print(f"   Dashboard: {'activado' if SHOW_DASHBOARD else 'desactivado'} | HUD: {'debug' if HUD_DEBUG else 'público'}")
print("➡️ Siguiente paso: prepará el entorno y cargá el bundle.")

### Elegí el flujo por lo que querés hacer

| Objetivo | PostgreSQL | Revisión | Resultado principal |
|---|---:|---:|---|
| Probar el modelo | No | No | Video + estados en memoria |
| Generar feedback portable | No | Sí | Video + paquete HITL |
| Guardar inferencias | Sí | No | Features y predicciones |
| Guardar y revisar | Sí | Sí | Inferencias + validaciones + paquete HITL |

<details>
<summary><strong>Ver las cuatro recetas completas</strong></summary>

#### A. Piloto offline recomendado
```python
ALLOW_PILOT_BUNDLE = True
ALLOW_EXPERIMENTAL_BUNDLE = False
PERSIST_TO_DATABASE = False
ENABLE_HUMAN_REVIEW = False
REVIEW_MODE = "priority"
DOWNLOAD_ANNOTATED_VIDEO = True
SHOW_DASHBOARD = True
HUD_DEBUG = False
```
No necesita Secrets. Genera el video y muestra los estados sin escribir datos remotos.

#### B. Piloto con HITL portable
```python
ALLOW_PILOT_BUNDLE = True
ALLOW_EXPERIMENTAL_BUNDLE = False
PERSIST_TO_DATABASE = False
ENABLE_HUMAN_REVIEW = True
REVIEW_MODE = "priority"
DOWNLOAD_ANNOTATED_VIDEO = True
SHOW_DASHBOARD = True
HUD_DEBUG = False
```
Necesita `VAAET_REVIEWER_ID`. Al finalizar la revisión ejecutá `finalize_current_review()`.

#### C. Persistencia sin revisión
```python
ALLOW_PILOT_BUNDLE = True
ALLOW_EXPERIMENTAL_BUNDLE = False
PERSIST_TO_DATABASE = True
ENABLE_HUMAN_REVIEW = False
REVIEW_MODE = "priority"
DOWNLOAD_ANNOTATED_VIDEO = True
SHOW_DASHBOARD = True
HUD_DEBUG = False
```
Necesita el perfil `inference`. Guarda features y predicciones, pero no crea etiquetas humanas.

#### D. PostgreSQL + revisión prioritaria
```python
ALLOW_PILOT_BUNDLE = True
ALLOW_EXPERIMENTAL_BUNDLE = False
PERSIST_TO_DATABASE = True
ENABLE_HUMAN_REVIEW = True
REVIEW_MODE = "priority"
DOWNLOAD_ANNOTATED_VIDEO = True
SHOW_DASHBOARD = True
HUD_DEBUG = False
```
Necesita los perfiles `inference`, `review` y `VAAET_REVIEWER_ID`. Guarda decisiones append-only y crea también el paquete HITL.

</details>

<details>
<summary><strong>Opciones independientes y casos especiales</strong></summary>

- `REVIEW_MODE="priority"`: muestra incidentes posibles, baja confianza y transiciones. Es la opción recomendada.
- `REVIEW_MODE="all"`: permite revisar todos los minutos clasificables.
- `ALLOW_EXPERIMENTAL_BUNDLE=True`: permite probar un `candidate`, siempre offline; PostgreSQL queda bloqueado.
- Un bundle `production` no necesita autorización especial, pero PostgreSQL sigue dependiendo de `PERSIST_TO_DATABASE`.
- `HUD_DEBUG=True` agrega IDs y señales técnicas; `False` genera el video público.
- Menos de 60 segundos: sólo video. Entre 60 y 119: video + telemetría, todavía sin estado. Desde 120 segundos: primera clasificación estable.

</details>

La configuración completa de Secrets y TLS está en la [guía canónica de Colab](../../docs/operations/colab-guide.md#secrets-y-postgresql). Tener credenciales cargadas nunca activa escrituras.

## 2. Preparar el entorno y cargar el bundle

La celda instala VAAET, valida el origen del paquete y busca el bundle localmente, en Google Drive o mediante upload. Luego verifica manifiesto, checksums, las 19 features, tres salidas MLP y los cuatro estados públicos.

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
REPO_DIR = Path("/content/vaaet")

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR.resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("No se encontró la raíz del repositorio VAAET.")

os.chdir(REPO_ROOT)

def validate_runtime_version(version: tuple[int, int]) -> None:
    if not (3, 10) <= version <= (3, 13):
        raise RuntimeError(
            f"Unsupported Python {version[0]}.{version[1]}. VAAET supports Python 3.10–3.13. "
            "In Colab, select a compatible runtime such as 2026.07 (Python 3.12.13)."
        )

def install_project(command: list[str], *, extras: str) -> None:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode == 0:
        print(f"✅ Instalación de VAAET terminada | extras={extras}")
        return
    print("🔴 Falló la instalación de VAAET")
    print("----- pip stdout -----")
    print(result.stdout.strip() or "(empty)")
    print("----- pip stderr -----")
    print(result.stderr.strip() or "(empty)")
    raise RuntimeError(
        f"VAAET installation failed under Python {sys.version.split()[0]} with extras={extras}. "
        "Update the repository and re-run this cell. In Colab, runtime 2026.07 "
        "(Python 3.12.13) is the temporary fallback."
    )

CURRENT_PYTHON = (sys.version_info.major, sys.version_info.minor)
validate_runtime_version(CURRENT_PYTHON)
print(f"🐍 Runtime Python {sys.version.split()[0]} | supported: 3.10–3.13")
WORKFLOW_EXTRAS = "vision,training,visualization,database"
project_requirement = f"{REPO_ROOT}[{WORKFLOW_EXTRAS}]"
install_command = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    install_command.append(project_requirement)
else:
    install_command.extend(["-e", project_requirement])
install_project(install_command, extras=WORKFLOW_EXTRAS)

for module_name in tuple(sys.modules):
    if module_name == "vaaet" or module_name.startswith("vaaet."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import vaaet

def validate_vaaet_origin(package: object, repo_root: Path, in_colab: bool) -> Path:
    package_file = getattr(package, "__file__", None)
    if not package_file:
        package_path = list(getattr(package, "__path__", ()))
        raise ImportError(
            "The 'vaaet' import resolved to a namespace package instead of the installed package. "
            f"Resolved locations: {package_path}. Re-run this setup cell."
        )
    origin = Path(package_file).resolve()
    expected_editable_root = (repo_root / "src/vaaet").resolve()
    if in_colab and repo_root.resolve() in origin.parents:
        raise ImportError(f"Colab debe cargar el paquete instalado, no la carpeta del repositorio: {origin}")
    if not in_colab and origin.parent != expected_editable_root:
        raise ImportError(f"La instalación editable local tiene un origen inesperado: {origin}")
    return origin

VAAET_PACKAGE_FILE = validate_vaaet_origin(vaaet, REPO_ROOT, IN_COLAB)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    check=False,
)
pip_check_output = "\n".join(
    part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
)
if pip_check.returncode == 0:
    print("✅ pip check: dependencias consistentes")
else:
    print("⚠️ pip check detectó conflictos en el runtime administrado:")
    print(pip_check_output or "No se recibió un diagnóstico")
    print("ℹ️ Se continúa porque los imports del flujo se validan a continuación.")

def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not required"

print({name: package_version(name) for name in ("numpy", "tensorflow", "opencv-python-headless", "ultralytics-opencv-headless")})

import os
import shutil

import cv2
import joblib
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import tensorflow as tf
import ultralytics

from vaaet.artifacts import MANIFEST_FILE, validate_manifest
from vaaet.data.database import DatabaseProfile, database_engine, get_optional_database_settings, inspect_database, load_reviewer_id
from vaaet.data.persistence import persist_classified_telemetry
from vaaet.data.pipeline_runs import PipelineRunMetadata, PipelineWorkflow, pipeline_run
from vaaet.data.review import build_review_widget, finalize_review_session, load_review_queue, persist_human_validation, select_review_queue
from vaaet.evaluation.reporting import show_inference_dashboard
from vaaet.inference.traffic_state import classify_raw_telemetry
from vaaet.logging import configure_logging
from vaaet.settings import DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABEL_MAP_PATH, MODEL_DIR, MODEL_PATH, RANDOM_SEED, SCALER_PATH, STATE_LABELS
from vaaet.vision.analysis import TrafficStatePrediction, analyze_video
from vaaet.vision.hud import HudConfig

configure_logging()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | TensorFlow {tf.__version__} | GPU {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Paquete: {VAAET_PACKAGE_FILE}")
GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f"✅ Flujo de inferencia listo | root={REPO_ROOT} | commit={GIT_COMMIT}")

_model_dir_abs = REPO_ROOT / MODEL_DIR
_model_dir_abs.mkdir(parents=True, exist_ok=True)
_ARTIFACT_NAMES = [Path(MODEL_PATH).name, Path(SCALER_PATH).name, Path(LABEL_MAP_PATH).name, MANIFEST_FILE]

def _bundle_paths(directory: Path) -> dict[str, Path]:
    return {name: directory / name for name in _ARTIFACT_NAMES}

paths = _bundle_paths(_model_dir_abs)
source = "local"
if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        drive_paths = _bundle_paths(Path("/content/drive") / DRIVE_ARTIFACT_DIR)
        if all(path.is_file() for path in drive_paths.values()):
            for name, path in drive_paths.items():
                shutil.copy2(path, paths[name])
            source = "Google Drive"
    except Exception as exc:
        print(f"⚠️ Google Drive no está disponible: {type(exc).__name__}")

if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    from google.colab import files

    print("📤 Subí los cuatro archivos completos del bundle:", _ARTIFACT_NAMES)
    for name, content in files.upload().items():
        if name in _ARTIFACT_NAMES:
            paths[name].write_bytes(content)
    source = "upload"

if all(path.is_file() for path in paths.values()):
    manifest = validate_manifest(_model_dir_abs)
    DEPLOYMENT_STAGE = manifest["training_lifecycle"]["deployment_stage"]
    MODEL_INPUT_POLICY = manifest["training_lifecycle"]["input_policy"]
    if DEPLOYMENT_STAGE == "pilot" and not ALLOW_PILOT_BUNDLE:
        raise RuntimeError("El bundle piloto no está autorizado. Usá ALLOW_PILOT_BUNDLE=True de forma explícita.")
    if DEPLOYMENT_STAGE == "candidate" and not ALLOW_EXPERIMENTAL_BUNDLE:
        raise RuntimeError("El bundle candidato no está aprobado. Autorizalo sólo para una evaluación offline explícita.")
    if DEPLOYMENT_STAGE == "candidate" and PERSIST_TO_DATABASE:
        raise RuntimeError("Los bundles candidatos son sólo offline. Usá PERSIST_TO_DATABASE=False.")
    model = tf.keras.models.load_model(paths[Path(MODEL_PATH).name])
    scaler = joblib.load(paths[Path(SCALER_PATH).name])
    label_mapping = joblib.load(paths[Path(LABEL_MAP_PATH).name])
    if dict(label_mapping) != dict(STATE_LABELS):
        raise RuntimeError("label_mapping.joblib no coincide con los cuatro estados públicos.")
    if int(model.output_shape[-1]) != 3:
        raise RuntimeError("El MLP del bundle debe exponer exactamente tres estados estables.")
    if int(getattr(scaler, "n_features_in_", -1)) != len(FEATURE_COLS):
        raise RuntimeError("El scaler del bundle no coincide con el contrato de 19 features.")
    hitl_destination = (
        "PostgreSQL + paquete inmutable"
        if ENABLE_HUMAN_REVIEW and PERSIST_TO_DATABASE
        else "paquete portable inmutable"
        if ENABLE_HUMAN_REVIEW
        else "desactivado"
    )
    print(f"✅ Bundle {DEPLOYMENT_STAGE.upper()} válido, cargado desde {source} | input_policy={MODEL_INPUT_POLICY}")
    print("\n🧭 Flujo de inferencia confirmado")
    print(f"   Etapa del bundle: {DEPLOYMENT_STAGE}")
    print(f"   PostgreSQL: {'activado' if PERSIST_TO_DATABASE else 'desactivado'}")
    print(f"   Revisión humana: {REVIEW_MODE if ENABLE_HUMAN_REVIEW else 'desactivada'}")
    print(f"   Destino HITL: {hitl_destination}")
    if DEPLOYMENT_STAGE == "pilot":
        print("   ⚠️ El bundle es piloto: sirve para iniciar el ciclo, pero no está aprobado para producción.")
    elif DEPLOYMENT_STAGE == "candidate":
        print("   ⚠️ El bundle es candidato experimental: la persistencia operacional está bloqueada.")
else:
    missing = [name for name, path in paths.items() if not path.is_file()]
    raise FileNotFoundError(f"El bundle está incompleto; faltan: {missing}")


## 3. Seleccionar el video

En Colab se solicita un MP4 mediante upload. En desarrollo local se intenta usar `data/sample/sample.mp4`. El nombre recomendado es `bridge_YYYY-MM-DD_HH-MM-SS_to_HH-MM-SS.mp4` para conservar trazabilidad temporal.

In [ ]:
VIDEO_PATH: Path | None = None
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded))).resolve()
else:
    candidate = REPO_ROOT / "data/sample/sample.mp4"
    VIDEO_PATH = candidate if candidate.is_file() else None

print(f"Video seleccionado: {VIDEO_PATH or 'ninguno'}")
print("➡️ Siguiente paso: analizá el video." if VIDEO_PATH else "⚠️ Subí o seleccioná un MP4 para continuar.")

## 4. Procesar y clasificar el video

El video se procesa completo y el contrato produce una fila raw por cada ventana completa de 60 segundos. El primer minuto es la línea base de las features temporales: se necesitan dos minutos consecutivos para obtener la primera fila clasificable. La clasificación final ya se ejecuta en esta celda; no es necesario volver a calcularla después.

In [ ]:
def prediction_provider(raw: pd.DataFrame) -> TrafficStatePrediction | None:
    try:
        classified = classify_raw_telemetry(
            raw,
            model,
            scaler,
            label_mapping=label_mapping,
            feature_cols=FEATURE_COLS,
            model_version=manifest["model_version"],
            input_policy=MODEL_INPUT_POLICY,
            inference_mode="stable",
            decision_policy=manifest["decision_policy"],
        )
    except ValueError:
        return None
    if classified.empty:
        return None
    latest = classified.iloc[-1]
    return TrafficStatePrediction(
        state=int(latest["traffic_state"]),
        label=str(latest["state_label"]),
        confidence=float(latest["confidence"]),
        evidence=float(latest.get("accident_evidence_score", 0.0)),
        incident_candidate=bool(latest.get("accident_rule_triggered", False)),
    )

if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Seleccioná o subí un MP4 válido antes de continuar.")

OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4" if IN_COLAB else VIDEO_PATH.with_name(f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4")
_run_metadata = PipelineRunMetadata(workflow=PipelineWorkflow.INFERENCE, git_commit=GIT_COMMIT, source_kind="video", clip_id=VIDEO_PATH.stem, model_version=manifest["model_version"])
with pipeline_run(_run_metadata, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs") as _run:
    analysis_result = analyze_video(
        VIDEO_PATH,
        OUTPUT_VIDEO,
        prediction_provider=prediction_provider,
        hud_config=HudConfig(debug=HUD_DEBUG),
    )
    _run.set_output_rows(len(analysis_result.telemetry))
LOCAL_INFERENCE_RUN_ID = str(_run.id)
df_telemetry = analysis_result.telemetry
df_classified = None
INFERENCE_PIPELINE_RUN_ID = None
REVIEW_VALIDATIONS = []
REVIEW_EXPORT_FRAME = None
if df_telemetry.empty:
    print("ℹ️ El video se procesó bien, pero no contiene un minuto completo.")
    print("   Se omiten features, clasificación, PostgreSQL y revisión HITL.")
    print(f"   Duración procesada: {analysis_result.processed_duration_seconds:.1f}s | mínimo: 60.0s")
else:
    df_classified = classify_raw_telemetry(
        df_telemetry,
        model,
        scaler,
        label_mapping=label_mapping,
        feature_cols=FEATURE_COLS,
        model_version=manifest["model_version"],
        input_policy=MODEL_INPUT_POLICY,
        inference_mode="stable",
        decision_policy=manifest["decision_policy"],
    )
    if df_classified.empty:
        print("ℹ️ Hay telemetría, pero todavía falta contexto para un estado estable.")
        print("   Se necesitan dos ventanas consecutivas de 60 segundos.")
        print("   Se omiten métricas, PostgreSQL, revisión HITL y dashboard.")
    else:
        display(df_classified)
        print("✅ Minutos clasificados por estado:")
        for code in sorted(df_classified["traffic_state"].unique()):
            count = int(df_classified["traffic_state"].eq(code).sum())
            print(f"   {STATE_LABELS.get(int(code), 'Desconocido'):>10}: {count} minutos")
        automatic_accidents = int(df_classified["traffic_state"].eq(3).sum())
        incident_candidates = int(
            df_classified.get("accident_alert_started", pd.Series(False, index=df_classified.index)).sum()
        )
        print(f"   Accident automáticos: {automatic_accidents} (siempre debe ser cero)")
        print(f"   Posibles incidentes: {incident_candidates} (el estado permanece Congested)")
    print(f"✅ Minutos de telemetría: {analysis_result.complete_minutes} | minutos clasificables: {len(df_classified)} | tramo descartado: {analysis_result.discarded_partial_seconds:.1f}s")
print(f"✅ Video anotado: {analysis_result.video_path}")
print("➡️ Siguiente paso: revisá PostgreSQL y HITL según la configuración elegida.")

if IN_COLAB and DOWNLOAD_ANNOTATED_VIDEO:
    from google.colab import files

    files.download(str(analysis_result.video_path))
elif IN_COLAB:
    print("ℹ️ Descarga automática desactivada; el video queda en /content.")

### Resultado de la clasificación

La celda anterior ya calculó `df_classified` mediante la cadena compartida `features → scaler → MLP → calibración → política temporal → detector de posible incidente`. No vuelvas a clasificar manualmente el mismo clip.

`traffic_state` sólo puede ser `Normal`, `Reduced` o `Congested`. Un posible accidente conserva `Congested` y activa `accident_rule_triggered`; `Accident` únicamente puede surgir de una validación humana.

## 5. Guardar en PostgreSQL (opcional)

Al activarla, el perfil `inference` escribe únicamente features y predicciones. Si la conexión, migración o permisos fallan, `df_classified` permanece disponible y el video no se pierde.

In [ ]:
# Cell 5 — Optional PostgreSQL persistence
inference_settings = (
    get_optional_database_settings(DatabaseProfile.INFERENCE) if PERSIST_TO_DATABASE else None
)

if inference_settings is not None:
    try:
        if df_classified is not None and not df_classified.empty:
            with database_engine(inference_settings) as db_engine:
                health = inspect_database(db_engine, DatabaseProfile.INFERENCE)
                print(f"PostgreSQL {health.server_version} | TLS={health.ssl_enabled} | schemas={health.available_schemas}")
                persisted = persist_classified_telemetry(
                    df_classified, engine=db_engine, model_version=manifest["model_version"]
                )
            INFERENCE_PIPELINE_RUN_ID = persisted.pipeline_run_id
            print(f"✅ PostgreSQL: {persisted.telemetry_rows} filas de features | {persisted.classification_rows} predicciones | run={INFERENCE_PIPELINE_RUN_ID}")
        else:
            print("ℹ️ PostgreSQL omitido: no hay minutos clasificables.")
    except NameError:
        print("⚠️ Primero ejecutá el análisis del video y después repetí esta celda.")
    except Exception as error:
        print(f"🔴 Falló PostgreSQL ({type(error).__name__}). Revisá migración, TLS y permisos del rol.")
        print("   Los datos clasificados siguen disponibles en df_classified.")
elif PERSIST_TO_DATABASE:
    print("⚠️ Activaste PostgreSQL, pero el perfil inference no está disponible.")
    print("   Configurá el perfil según la guía canónica de Colab.")
    print("   Las filas siguen disponibles en df_classified.")
else:
    print("ℹ️ PostgreSQL desactivado por la configuración central.")
    print("   Las filas siguen en df_classified y pueden entrar en una revisión HITL portable.")
print("➡️ Siguiente paso: iniciá la revisión humana si la habilitaste.")


## 6. Revisión humana HITL

La revisión se abre **después** de terminar el clip; no interrumpe la inferencia con pop-ups. Mirá el video anotado, buscá el `record_time` indicado y contrastá también los minutos anterior y posterior.

- **Confirmar:** conservar el estado predicho.
- **Corregir:** elegir `Normal`, `Reduced` o `Congested`.
- **Omitir:** la fila queda pendiente y nunca se usa como ground truth.
- **Confirmar Accident:** elegir `Accident`, marcar `I reviewed temporal context` y escribir una nota. El estado automático permanece `Congested`; Accident sólo existe como decisión humana.

Con PostgreSQL, `Save validation` inserta una decisión append-only en `vaaet_feedback.human_validations`; nunca edites la predicción manualmente. Sin PostgreSQL, las decisiones quedan en el paquete portable. En ambos casos, al terminar ejecutá `finalize_current_review()` para cerrar la sesión de forma idempotente. El reentrenamiento se realiza exclusivamente en `train_traffic_state_classifier.ipynb`.


In [ ]:
# Cell 6 — Explicit human review after inference
if ENABLE_HUMAN_REVIEW:
    reviewer_id = load_reviewer_id()
    review_settings = get_optional_database_settings(DatabaseProfile.REVIEW)
    REVIEW_VALIDATIONS = []
    if review_settings is not None and INFERENCE_PIPELINE_RUN_ID is not None:
        full_review_queue = load_review_queue(
            settings=review_settings, pipeline_run_id=INFERENCE_PIPELINE_RUN_ID, mode="all"
        )
        review_queue = select_review_queue(full_review_queue, mode=REVIEW_MODE)
        print(f"✅ Cola PostgreSQL: {len(review_queue)} de {len(full_review_queue)} filas seleccionadas ({REVIEW_MODE}).")
        _prediction_keys = full_review_queue[["clip_id", "record_time", "prediction_id"]].copy()
        _prediction_keys["record_time"] = pd.to_datetime(_prediction_keys["record_time"], utc=True)
        REVIEW_EXPORT_FRAME = df_classified.copy()
        REVIEW_EXPORT_FRAME["record_time"] = pd.to_datetime(REVIEW_EXPORT_FRAME["record_time"], utc=True)
        REVIEW_EXPORT_FRAME = REVIEW_EXPORT_FRAME.merge(_prediction_keys, on=["clip_id", "record_time"], how="left", validate="one_to_one")
        if REVIEW_EXPORT_FRAME["prediction_id"].isna().any():
            raise RuntimeError("La cola de revisión PostgreSQL no cubre todas las filas inferidas.")
        def _persist_and_accumulate(decision):
            persist_human_validation(decision, settings=review_settings)
            REVIEW_VALIDATIONS.append(decision)
        build_review_widget(review_queue, reviewer_id=reviewer_id, on_submit=_persist_and_accumulate)
    elif df_classified is not None and not df_classified.empty:
        reason = (
            "la inferencia no se persistió"
            if INFERENCE_PIPELINE_RUN_ID is None
            else "el perfil review no está disponible"
        )
        print(f"ℹ️ Revisión portable: {reason}.")
        print("   Las decisiones no modifican PostgreSQL y quedan en el paquete inmutable.")
        REVIEW_EXPORT_FRAME = df_classified.copy().reset_index(drop=True)
        REVIEW_EXPORT_FRAME["prediction_id"] = REVIEW_EXPORT_FRAME.index + 1
        local_queue = select_review_queue(REVIEW_EXPORT_FRAME, mode=REVIEW_MODE)
        print(f"✅ Cola portable: {len(local_queue)} de {len(REVIEW_EXPORT_FRAME)} filas seleccionadas ({REVIEW_MODE}).")
        build_review_widget(local_queue, reviewer_id=reviewer_id, on_submit=REVIEW_VALIDATIONS.append)
    else:
        print("ℹ️ Revisión HITL omitida: no hay minutos clasificables.")

    if REVIEW_EXPORT_FRAME is not None:
        def finalize_current_review():
            canonical_root = None
            if IN_COLAB:
                from google.colab import drive  # type: ignore[import-untyped]
                drive.mount("/content/drive", force_remount=False)
                canonical_root = Path("/content/drive/MyDrive/vaaet-ml/data/hitl-reviews")
            result = finalize_review_session(
                classified=REVIEW_EXPORT_FRAME,
                validations=REVIEW_VALIDATIONS,
                pipeline_run_id=INFERENCE_PIPELINE_RUN_ID or LOCAL_INFERENCE_RUN_ID,
                model_version=manifest["model_version"],
                git_commit=GIT_COMMIT,
                vaaet_version=package_version("vaaet-ml"),
                local_root=REPO_ROOT / "data/processed/hitl-reviews",
                canonical_root=canonical_root,
            )
            print(f"📦 HITL package {result.package_id} | reviewed={result.reviewed_rows} | pending={result.pending_rows}")
            print(f"   fingerprint={result.fingerprint} | status={result.sync_status}")
            print(f"   location={result.canonical_path or result.local_path}")
            if result.sync_error:
                print(f"⚠️ Falló la sincronización con Drive; el paquete local queda pendiente: {result.sync_error}")
            return result
        print("Después de revisar u omitir filas, ejecutá finalize_current_review() una vez. Repetirlo es idempotente.")
else:
    print("ℹ️ Revisión humana desactivada. Para el próximo video usá ENABLE_HUMAN_REVIEW=True.")


## 7. Ver el resumen visual

El dashboard resume distribución de estados, velocidad, confianza, tipos de vehículos y relación velocidad/volumen. Sólo se ejecuta cuando `SHOW_DASHBOARD=True` y existen minutos clasificados.

In [ ]:
# Cell 7 — Optional visualization dashboard
if not SHOW_DASHBOARD:
    print("ℹ️ Dashboard desactivado por la configuración central.")
elif 'df_classified' not in globals():
    print("⚠️ Primero ejecutá el análisis del video.")
elif df_classified is None or df_classified.empty:
    print("ℹ️ No hay minutos clasificables para mostrar. Se necesitan dos minutos completos consecutivos.")
else:
    show_inference_dashboard(df_classified, state_labels=label_mapping)
    print("✅ Dashboard generado.")
    print("➡️ Fin del flujo: conservá el video y, si revisaste filas, finalizá el paquete HITL.")